In [1]:
# -*- coding: utf-8 -*-
"""
K=438 코사인 K-means → Parquet + NPY 저장 스크립트
- 입력: /kaggle/working/seq_vec.npz (keys: "seqs", "X")
- 출력 디렉토리: /kaggle/working/latte_clusters/k438
  ├─ meta.json
  ├─ clusters.parquet          # 438행 (클러스터 요약)
  ├─ members.parquet           # N행 (sequence → cluster_id 매핑)
  ├─ centroids.npy             # (438, D) float32
  ├─ center_vecs.npy           # (438, D) float32
"""

import os, json, datetime
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd


# =========================
# 0) IO 유틸
# =========================
def load_npz(path: str = "/kaggle/working/seq_vec.npz") -> Tuple[List[str], np.ndarray]:
    data = np.load(path, allow_pickle=True)
    seqs = data["seqs"].tolist()
    X = data["X"]
    return seqs, X

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def make_cluster_id(k:int, cid:int) -> str:
    # 예: K0438-C0123
    return f"K{str(k).zfill(4)}-C{str(cid).zfill(4)}"


# =========================
# 1) 코사인 K-means
#    (X는 한 번만 L2 row-normalize)
# =========================
def l2_normalize_rows(X: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    nrm = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.clip(nrm, eps, None)

def _init_centroids_from_data(Xn: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    N = Xn.shape[0]
    if k > N:
        raise ValueError("k > N")
    idx = rng.choice(N, size=k, replace=False)
    return Xn[idx].copy()

def _assign(Xn: np.ndarray, C: np.ndarray) -> np.ndarray:
    # argmax dot == 최소 (1 - cos)
    return np.argmax(Xn @ C.T, axis=1)

def _update(Xn: np.ndarray, labels: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    D = Xn.shape[1]
    C = np.zeros((k, D), dtype=Xn.dtype)
    for c in range(k):
        members = Xn[labels == c]
        if members.size == 0:
            C[c] = Xn[rng.integers(0, Xn.shape[0])]
        else:
            v = members.mean(axis=0)
            n = np.linalg.norm(v)
            C[c] = v / n if n > 0 else v
    return C

def cosine_kmeans_full(
    Xn: np.ndarray,
    k: int,
    max_iter: int = 30,
    seed: int = 42,
    tol: float = 1e-4,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    C = _init_centroids_from_data(Xn, k, rng)
    for _ in range(max_iter):
        labels = _assign(Xn, C)
        C_new = _update(Xn, labels, k, rng)
        shift = np.linalg.norm(C_new - C, axis=1).max()
        C = C_new
        if shift < tol:
            break
    return labels, C

def cosine_kmeans_best_of(
    Xn: np.ndarray,
    k: int,
    n_init: int = 2,
    max_iter: int = 30,
    seed_base: int = 42,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    n_init 번 시도해서 평균 (1 - cos) 비용이 가장 낮은 run 선택
    """
    best_cost = float("inf")
    best_labels = None
    best_C = None
    for rep in range(n_init):
        labels, C = cosine_kmeans_full(Xn, k, max_iter=max_iter, seed=seed_base + rep)
        cost = mean_cosine_distance(Xn, labels, C)
        if cost < best_cost:
            best_cost = cost
            best_labels = labels
            best_C = C
    return best_labels, best_C

def mean_cosine_distance(Xn: np.ndarray, labels: np.ndarray, C: np.ndarray) -> float:
    sims = np.einsum("nd,nd->n", Xn, C[labels])
    return float(1.0 - sims.mean())


# =========================
# 2) 요약 (센터/센트로이드/멤버)
# =========================
def summarize_clusters(
    seqs: List[str],
    X_raw: np.ndarray,      # (N, D) 정규화 전 원본
    Xn: np.ndarray,         # (N, D) L2 정규화
    labels: np.ndarray,     # (N,)
    centroids_unit: np.ndarray,  # (k, D) unit-norm
) -> Dict[int, dict]:
    """
    return:
      { cid: {
          "center_idx": int,
          "center_seq": str,
          "center_vec": (D,) float32,
          "centroid_vec": (D,) float32,  # 원공간 평균
          "members": List[str]
        }, ... }
    """
    k = centroids_unit.shape[0]
    out = {}
    for c in range(k):
        idxs = np.where(labels == c)[0]
        if idxs.size == 0:
            out[c] = {
                "center_idx": None,
                "center_seq": None,
                "center_vec": None,
                "centroid_vec": None,
                "members": [],
            }
            continue

        # 원공간 평균(정규화 X)
        centroid_vec = X_raw[idxs].mean(axis=0).astype(np.float32)

        # 대표(centroid_unit과 가장 유사한 실제 멤버)
        sims = Xn[idxs] @ centroids_unit[c]
        best_local = int(np.argmax(sims))
        real_idx = int(idxs[best_local])

        out[c] = {
            "center_idx": real_idx,
            "center_seq": seqs[real_idx],
            "center_vec": X_raw[real_idx].astype(np.float32),
            "centroid_vec": centroid_vec,
            "members": [seqs[i] for i in idxs],
        }
    return out


# =========================
# 3) 저장 (Parquet + NPY + meta.json)
# =========================
def save_as_parquet_npy(
    clusters: Dict[int, dict],
    k: int,
    out_dir: str,
    N: int,
    D: int,
    distance_desc: str = "cosine (L2-normalized dot)",
    algo_desc: str = "cosine k-means; X row-normalized; centroids unit-norm",
    seed: int = 42,
    n_init: int = 2,
    max_iter_full: int = 30,
    cost_metric: str = "mean(1 - cos(x, centroid[label]))",
):
    ensure_dir(out_dir)

    cids_sorted = sorted(clusters.keys())

    centroid_vecs = []
    center_vecs = []
    center_seqs = []
    center_idxs = []
    n_members = []
    cluster_ids = []
    members_rows = []

    for cid in cids_sorted:
        info = clusters[cid]
        cluster_id = make_cluster_id(k, cid)
        cluster_ids.append(cluster_id)

        centroid_vecs.append(info["centroid_vec"].astype(np.float32))
        center_vecs.append(info["center_vec"].astype(np.float32))
        center_seqs.append(info["center_seq"])
        center_idxs.append(int(info["center_idx"]))
        mems = info["members"]
        n_members.append(len(mems))

        for s in mems:
            members_rows.append({"sequence": s, "cluster_id": cluster_id, "cid": int(cid)})

    centroid_mat = np.stack(centroid_vecs, axis=0)  # (k, D)
    center_mat   = np.stack(center_vecs, axis=0)    # (k, D)

    np.save(os.path.join(out_dir, "centroids.npy"), centroid_mat)
    np.save(os.path.join(out_dir, "center_vecs.npy"), center_mat)

    df_clusters = pd.DataFrame({
        "cluster_id": cluster_ids,
        "cid": cids_sorted,
        "center_seq": center_seqs,
        "center_idx": center_idxs,
        "center_vec_path": ["center_vecs.npy"] * len(cids_sorted),
        "centroid_vec_path": ["centroids.npy"] * len(cids_sorted),
        "n_members": n_members,
    })
    df_clusters.to_parquet(os.path.join(out_dir, "clusters.parquet"), index=False)

    # members는 큼: 필요시 partitioning 가능 (ex: partition by cid range)
    df_members = pd.DataFrame(members_rows)
    df_members.to_parquet(os.path.join(out_dir, "members.parquet"), index=False)

    meta = {
        "k": int(k),
        "N": int(N),
        "D": int(D),
        "distance": distance_desc,
        "algorithm": algo_desc,
        "seed": int(seed),
        "n_init": int(n_init),
        "max_iter_full": int(max_iter_full),
        "cost_metric": cost_metric,
        "created_at": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"[OK] Saved to: {out_dir}")


# =========================
# 4) 메인: 돌리고 저장
# =========================
def main():
    IN_PATH  = "/kaggle/input/latent-vec-calculation/seq_vec.npz"
    OUT_DIR  = "/kaggle/working/clusters_438"
    K        = 438
    N_INIT   = 3          # 안정성 높이고 싶으면 3
    MAX_ITER = 30
    SEED     = 42

    # 1) 로드
    seqs, X = load_npz(IN_PATH)
    N, D = X.shape
    print(f"[LOAD] N={N}, D={D}")

    # 2) 정규화(한 번만)
    Xn = l2_normalize_rows(X).astype(np.float32)

    # 3) 코사인 K-means (K=438)
    labels, centroids_unit = cosine_kmeans_best_of(
        Xn, k=K, n_init=N_INIT, max_iter=MAX_ITER, seed_base=SEED
    )
    print("[KM] done")

    # 4) 요약 산출
    clusters = summarize_clusters(
        seqs=seqs,
        X_raw=X,
        Xn=Xn,
        labels=labels,
        centroids_unit=centroids_unit,
    )
    print("[SUM] done")

    # 5) 저장 (Parquet + NPY + meta.json)
    save_as_parquet_npy(
        clusters=clusters,
        k=K,
        out_dir=OUT_DIR,
        N=N,
        D=D,
        seed=SEED,
        n_init=N_INIT,
        max_iter_full=MAX_ITER,
    )

if __name__ == "__main__":
    main()


[LOAD] N=873074, D=256
[KM] done
[SUM] done
[OK] Saved to: /kaggle/working/clusters_438
